# Cross-system trajectory integration

Integrates the frozen early Müller program across mouse, human Müller, human bulk retina, and organoid systems. Failed permutation prototypes were removed; only the final matched-null and concordance workflow is retained.

**Important:** These notebooks document the finalized analysis logic. Public raw data are downloaded from the cited repositories; large intermediate objects and outputs are intentionally not embedded in the GitHub version. Where a dataset lacks condition-level biological replication, results are explicitly treated as descriptive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd, json

ROOT = Path("/content/drive/MyDrive/DR_Muller_AdaptiveReactive")

with open(ROOT/"signatures/GSE328966_LOCKED_discovery_signatures.json") as f:
    sig = json.load(f)

for folder in [
    "01_GSE328966_early_mouse",
    "02_Human_scRNA_2026",
    "03_GSE160306_human_validation",
    "04_GSE313176_mouse_longitudinal",
    "05_GSE290024_organoid"
]:
    print(f"\n### {folder}")
    for p in sorted((ROOT/folder).rglob("*.csv*")):
        print(p.relative_to(ROOT))

In [ ]:
disc = pd.read_csv(ROOT/"01_GSE328966_early_mouse/GSE328966_Muller_early_late_trajectory.csv")
mouse_bulk = pd.read_csv(ROOT/"04_GSE313176_mouse_longitudinal/GSE313176_retina_bulk_DESeq2_all_timepoints.csv.gz")
mouse_sc = pd.read_csv(ROOT/"04_GSE313176_mouse_longitudinal/scRNA_3mo/GSE313176_PRIMARY_Muller_pseudobulk_gene_effects.csv")
human_sc = pd.read_csv(ROOT/"02_Human_scRNA_2026/Human_Muller_LOCKED_signature_exact_tests.csv")
human_bulk = pd.read_csv(ROOT/"03_GSE160306_human_validation/GSE160306_FINAL_locked_signature_tests_79samples.csv")
org = pd.read_csv(ROOT/"05_GSE290024_organoid/GSE290024_edgeR_DE_all_genes.csv")

for name, x in {
    "discovery":disc, "mouse_bulk":mouse_bulk, "mouse_scRNA":mouse_sc,
    "human_scRNA":human_sc, "human_bulk":human_bulk, "organoid":org
}.items():
    print("\n", name, x.shape)
    print(x.columns.tolist())

In [ ]:
early = pd.DataFrame({"mouse_gene": sig["early_up_transient_all"]})

# Discovery mouse
d = disc.rename(columns={"Unnamed: 0":"mouse_gene"})[
    ["mouse_gene","early_effect","late_effect"]
]

# Longitudinal mouse bulk
mb = (mouse_bulk[mouse_bulk["symbol"].isin(early.mouse_gene)]
      .pivot_table(index="symbol", columns="timepoint",
                   values="log2FoldChange", aggfunc="mean")
      .add_prefix("bulk_")
      .reset_index()
      .rename(columns={"symbol":"mouse_gene"}))

# 3mo mouse Müller scRNA
ms = mouse_sc.rename(columns={
    "Unnamed: 0":"mouse_gene",
    "log2FC_STZ_vs_Sham":"muller3mo"
})[["mouse_gene","muller3mo"]]

# Human organoid via locked ortholog map
orth = pd.read_csv(ROOT/"02_Human_scRNA_2026/Mouse_to_Human_orthologs_LOCKED.csv")
og = (orth.merge(org[["symbol","logFC"]], left_on="human_symbol", right_on="symbol")
      .groupby("mouse_symbol", as_index=False)["logFC"].mean()
      .rename(columns={"mouse_symbol":"mouse_gene","logFC":"organoid"}))

M = early.merge(d, on="mouse_gene", how="left") \
         .merge(mb, on="mouse_gene", how="left") \
         .merge(ms, on="mouse_gene", how="left") \
         .merge(og, on="mouse_gene", how="left")

print("Matrix:", M.shape)
print("\nCoverage:")
print(M.notna().sum())
display(M.head())

In [ ]:
from scipy.stats import spearmanr
import numpy as np, pandas as pd

cols = ["late_effect","bulk_1mo","bulk_3mo","bulk_6mo","muller3mo","organoid"]

rows = []
for c in cols:
    x = M[["early_effect", c]].dropna()
    rho, p = spearmanr(x["early_effect"], x[c])

    rows.append({
        "dataset": c,
        "n": len(x),
        "median_effect": x[c].median(),
        "pct_positive": 100*(x[c] > 0).mean(),
        "rho_vs_early": rho,
        "p_rho": p
    })

obs = pd.DataFrame(rows)
display(obs.round(3))

In [ ]:
# FULL mouse-bulk background — not restricted to Early184
mb_all = (
    mouse_bulk.dropna(subset=["symbol"])
    .pivot_table(index="symbol", columns="timepoint",
                 values="log2FoldChange", aggfunc="mean")
    .add_prefix("bulk_")
    .reset_index()
    .rename(columns={"symbol":"mouse_gene"})
)

# FULL 3mo Müller background
ms_all = mouse_sc.rename(columns={
    "Unnamed: 0":"mouse_gene",
    "log2FC_STZ_vs_Sham":"muller3mo"
})[["mouse_gene","muller3mo"]]

bg = (
    disc.rename(columns={"Unnamed: 0":"mouse_gene"})[
        ["mouse_gene","early_effect","trajectory_eligible"]
    ]
    .merge(mb_all, on="mouse_gene", how="left")
    .merge(ms_all, on="mouse_gene", how="left")
)

print("Background genes:", len(bg))
print(bg[["bulk_1mo","bulk_3mo","bulk_6mo","muller3mo"]].notna().sum())

In [ ]:
rng = np.random.default_rng(42)
early_set = set(M.mouse_gene)
rows = []

for c in ["bulk_1mo","bulk_3mo","bulk_6mo","muller3mo"]:

    t = bg[bg.mouse_gene.isin(early_set) & bg[c].notna()].copy()
    b = bg[~bg.mouse_gene.isin(early_set) &
           bg[c].notna() &
           bg.early_effect.notna()].copy()

    be, bv = b.early_effect.to_numpy(), b[c].to_numpy()

    # 50 nearest genes matched on discovery early-effect magnitude
    cand = []
    for e in t.early_effect:
        idx = np.argpartition(np.abs(be-e), 50)[:50]
        cand.append(bv[idx])
    cand = np.vstack(cand)

    null = np.empty(5000)
    for k in range(5000):
        pick = rng.integers(0, 50, size=len(t))
        null[k] = np.median(cand[np.arange(len(t)), pick])

    obs = np.median(t[c])
    center = np.median(null)
    p = (1 + np.sum(np.abs(null-center) >= abs(obs-center))) / 5001

    rows.append([c,len(t),obs,center,p])

perm_mouse = pd.DataFrame(rows, columns=[
    "dataset","n","Early184_median","matched_null_median","empirical_p"
])

display(perm_mouse.round(4))

In [ ]:
!pip -q install gprofiler-official

from gprofiler import GProfiler

genes = bg["mouse_gene"].dropna().unique().tolist()

gp = GProfiler(return_dataframe=True)

ortho = gp.orth(
    organism="mmusculus",
    target="hsapiens",
    query=genes
)

ortho = ortho[
    ["incoming","name"]
].dropna().drop_duplicates()

ortho.columns = ["mouse_gene","human_gene"]

print("Mouse genes mapped:", ortho.mouse_gene.nunique(), "/", len(genes))
print("Human orthologs:", ortho.human_gene.nunique())
display(ortho.head())

In [ ]:
# Map genome-wide mouse genes to organoid human effects
org_bg = (
    ortho.merge(org[["symbol","logFC"]], left_on="human_gene", right_on="symbol")
    .groupby("mouse_gene", as_index=False)["logFC"].mean()
    .rename(columns={"logFC":"organoid"})
)

bg_org = bg[["mouse_gene","early_effect"]].merge(org_bg, on="mouse_gene", how="left")

early_set = set(M.mouse_gene)

t = bg_org[
    bg_org.mouse_gene.isin(early_set) &
    bg_org.organoid.notna()
].copy()

b = bg_org[
    ~bg_org.mouse_gene.isin(early_set) &
    bg_org.organoid.notna() &
    bg_org.early_effect.notna()
].copy()

be, bv = b.early_effect.to_numpy(), b.organoid.to_numpy()

cand = []
for e in t.early_effect:
    idx = np.argpartition(np.abs(be-e), 50)[:50]
    cand.append(bv[idx])
cand = np.vstack(cand)

rng = np.random.default_rng(42)
null = np.empty(5000)

for k in range(5000):
    pick = rng.integers(0, 50, size=len(t))
    null[k] = np.median(cand[np.arange(len(t)), pick])

obs = np.median(t.organoid)
center = np.median(null)
p = (1 + np.sum(np.abs(null-center) >= abs(obs-center))) / 5001

print("Early184 mapped:", len(t))
print("Observed median:", round(obs,4))
print("Matched-null median:", round(center,4))
print("Empirical p:", round(p,4))

In [ ]:
# Target = EXACT locked organoid representation from earlier analysis
locked_target = M[["mouse_gene","early_effect","organoid"]].dropna()

# Background = genome-wide ortholog mapping
b = bg_org[
    ~bg_org.mouse_gene.isin(early_set) &
    bg_org.organoid.notna() &
    bg_org.early_effect.notna()
].copy()

be, bv = b.early_effect.to_numpy(), b.organoid.to_numpy()

cand = []
for e in locked_target.early_effect:
    idx = np.argpartition(np.abs(be-e), 50)[:50]
    cand.append(bv[idx])
cand = np.vstack(cand)

rng = np.random.default_rng(42)
null = np.empty(5000)

for k in range(5000):
    pick = rng.integers(0,50,size=len(locked_target))
    null[k] = np.median(cand[np.arange(len(locked_target)), pick])

obs = locked_target.organoid.median()
center = np.median(null)
p = (1 + np.sum(np.abs(null-center) >= abs(obs-center))) / 5001

print("Locked Early184 tested:", len(locked_target))
print("Observed median:", round(obs,4))
print("Matched-null median:", round(center,4))
print("Empirical p:", round(p,4))

In [ ]:
from statsmodels.stats.multitest import multipletests

# Add corrected organoid result
null_tests = pd.concat([
    perm_mouse,
    pd.DataFrame([{
        "dataset":"organoid",
        "n":len(locked_target),
        "Early184_median":obs,
        "matched_null_median":center,
        "empirical_p":p
    }])
], ignore_index=True)

null_tests["FDR_BH"] = multipletests(null_tests["empirical_p"], method="fdr_bh")[1]

OUT = ROOT/"06_CrossSpecies_Trajectory_Integration"
OUT.mkdir(exist_ok=True)

null_tests.to_csv(OUT/"Early184_matched_null_tests.csv", index=False)

display(null_tests.round(4))

In [ ]:
paths = {
    "mouse_bulk": ROOT/"04_GSE313176_mouse_longitudinal/GSE313176_LOCKED_signature_validation_exact_permutation.csv",
    "mouse_muller": ROOT/"04_GSE313176_mouse_longitudinal/scRNA_3mo/GSE313176_PRIMARY_Muller_locked_signature_summary.csv",
    "human_scRNA": ROOT/"02_Human_scRNA_2026/Human_Muller_LOCKED_signature_exact_tests.csv",
    "human_bulk": ROOT/"03_GSE160306_human_validation/GSE160306_FINAL_locked_signature_tests_79samples.csv",
    "organoid": ROOT/"05_GSE290024_organoid/GSE290024_direct_signature_tests.csv"
}

for name,p in paths.items():
    x = pd.read_csv(p)
    print(f"\n### {name} {x.shape}")
    print(x.columns.tolist())
    display(x.head())

In [ ]:
name_map = {
    "early_up_transient_all":"Early184",
    "early_IFN_alpha_beta":"IFN6",
    "early_retinoid_metabolism":"Retinoid5",
    "early_ECM_GAG_exploratory":"ECM13",
    "IFN":"IFN6", "Retinoid":"Retinoid5",
    "ECM_GAG":"ECM13", "Early166":"Early184"
}

mb = pd.read_csv(paths["mouse_bulk"])
mm = pd.read_csv(paths["mouse_muller"])
hs = pd.read_csv(paths["human_scRNA"])
hb = pd.read_csv(paths["human_bulk"])
og = pd.read_csv(paths["organoid"])

rows = []

# replicated mouse whole retina
for _,r in mb.iterrows():
    s = name_map.get(r.signature, r.signature)
    if s in ["Early184","IFN6","Retinoid5","ECM13"]:
        rows.append(["Mouse bulk", r.timepoint, s,
                     r.delta_STZ_minus_Sham, r.exact_p_two_sided])

# descriptive 3mo mouse Müller
for _,r in mm.iterrows():
    s = name_map.get(r.signature, r.signature)
    if s in ["Early184","IFN6","Retinoid5","ECM13"]:
        rows.append(["Mouse Müller scRNA","3mo",s,r.median_log2FC,np.nan])

# human Müller scRNA
for _,r in hs.iterrows():
    s = name_map.get(r.signature, r.signature)
    if s in ["Early184","IFN6","Retinoid5","ECM13"]:
        rows += [
            ["Human Müller scRNA","DM-NON",s,r.DM_minus_NON,r.p_DM_gt_NON],
            ["Human Müller scRNA","DM-DR", s,r.DM_minus_DR, r.p_DM_gt_DR]
        ]

# human bulk retina
for _,r in hb.iterrows():
    s = name_map.get(r.signature, r.signature)
    rows.append(["Human bulk retina",r.contrast,s,r.effect_meanZ,r.p])

# human organoid
for _,r in og.iterrows():
    rows.append(["Human organoid","25mM-17mM",r.signature,r.effect,r.p])

traj = pd.DataFrame(rows, columns=["system","contrast","signature","effect","p"])
traj.to_csv(OUT/"Locked_signature_cross_system_trajectory.csv", index=False)

print(traj.shape)
display(traj.head(20))

In [ ]:
locked = {
    "Early184": sig["early_up_transient_all"],
    "IFN6": sig["early_IFN_alpha_beta"],
    "Retinoid5": sig["early_retinoid_metabolism"],
    "ECM13": sig["early_ECM_GAG_exploratory"]
}

D = disc.rename(columns={"Unnamed: 0":"gene"}).set_index("gene")

extra = []
for s, genes in locked.items():
    g = [x for x in genes if x in D.index]
    extra += [
        ["Mouse discovery Müller","5D", s, D.loc[g,"early_effect"].median(), np.nan],
        ["Mouse discovery Müller","15D",s, D.loc[g,"late_effect"].median(),  np.nan]
    ]

traj = pd.concat([
    pd.DataFrame(extra, columns=traj.columns),
    traj
], ignore_index=True)

traj.to_csv(OUT/"Locked_signature_cross_system_trajectory.csv", index=False)

display(traj.head(8).round(3))

In [ ]:
# Direction only: -1 down, 0 ~flat, +1 up
plot_df = traj.copy()
plot_df["direction"] = np.where(plot_df["effect"] > 0.05, 1,
                        np.where(plot_df["effect"] < -0.05, -1, 0))

# Mark nominal significance where a p-value exists
plot_df["mark"] = np.where(
    plot_df["p"].notna() & (plot_df["p"] < 0.05), "*", ""
)

plot_df["context"] = plot_df["system"] + " | " + plot_df["contrast"]

direction_mat = plot_df.pivot(
    index="signature",
    columns="context",
    values="direction"
)

mark_mat = plot_df.pivot(
    index="signature",
    columns="context",
    values="mark"
)

print("Heatmap matrix:", direction_mat.shape)
display(direction_mat)

In [ ]:
order = [
    "Mouse discovery Müller | 5D",
    "Mouse discovery Müller | 15D",
    "Mouse bulk | 1mo",
    "Mouse bulk | 3mo",
    "Mouse Müller scRNA | 3mo",
    "Mouse bulk | 6mo",
    "Human Müller scRNA | DM-NON",
    "Human Müller scRNA | DM-DR",
    "Human organoid | 25mM-17mM",
    "Human bulk retina | DMvsCTRL_P",
    "Human bulk retina | DMvsCTRL_M",
    "Human bulk retina | NPDRvsCTRL_P",
    "Human bulk retina | NPDRvsCTRL_M",
    "Human bulk retina | ADVvsCTRL_P",
    "Human bulk retina | ADVvsCTRL_M",
    "Human bulk retina | NPDRvsDM_P",
    "Human bulk retina | NPDRvsDM_M",
    "Human bulk retina | ADVvsNPDR_P",
    "Human bulk retina | ADVvsNPDR_M"
]

direction_mat = direction_mat[order]
mark_mat = mark_mat[order]

direction_mat.to_csv(OUT/"Locked_signature_direction_matrix.csv")
mark_mat.to_csv(OUT/"Locked_signature_nominalP_marks.csv")

display(direction_mat)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

row_order = ["Early184","IFN6","Retinoid5","ECM13"]
X = direction_mat.loc[row_order]

fig, ax = plt.subplots(figsize=(14,3.6))
im = ax.imshow(X.values, aspect="auto",
               cmap=ListedColormap(["#3B6FB6","#F2F2F2","#C9473A"]),
               vmin=-1, vmax=1)

ax.set_yticks(range(len(row_order)))
ax.set_yticklabels(row_order, fontsize=10)

ax.set_xticks(range(len(X.columns)))
ax.set_xticklabels(
    [x.replace(" | ","\n") for x in X.columns],
    rotation=55, ha="right", fontsize=8
)

# nominal p<0.05 marks
for i,s in enumerate(row_order):
    for j,c in enumerate(X.columns):
        if mark_mat.loc[s,c] == "*":
            ax.text(j,i,"*",ha="center",va="center",
                    fontsize=13,fontweight="bold")

# separators between biological systems
for x in [1.5,5.5,7.5,8.5]:
    ax.axvline(x, color="black", lw=.8)

ax.set_title("Trajectory of locked early Müller programs across mouse and human systems",
             fontsize=12, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()

fig.savefig(OUT/"Figure_integrated_signature_trajectory.svg",
            bbox_inches="tight")
fig.savefig(OUT/"Figure_integrated_signature_trajectory.pdf",
            bbox_inches="tight")

plt.show()

In [ ]:
primary_order = [
    "Mouse discovery Müller | 5D",
    "Mouse discovery Müller | 15D",
    "Mouse bulk | 1mo",
    "Mouse bulk | 3mo",
    "Mouse Müller scRNA | 3mo",
    "Mouse bulk | 6mo",
    "Human Müller scRNA | DM-NON",
    "Human Müller scRNA | DM-DR",
    "Human organoid | 25mM-17mM",
    "Human bulk retina | DMvsCTRL_P",
    "Human bulk retina | DMvsCTRL_M",
    "Human bulk retina | NPDRvsCTRL_P",
    "Human bulk retina | NPDRvsCTRL_M",
    "Human bulk retina | ADVvsCTRL_P",
    "Human bulk retina | ADVvsCTRL_M"
]

P = plot_df[plot_df["context"].isin(primary_order)].copy()

# Direction ONLY — no arbitrary magnitude threshold
P["direction"] = np.sign(P["effect"]).astype(int)

primary_mat = P.pivot(index="signature", columns="context",
                      values="direction")[primary_order]

primary_marks = P.pivot(index="signature", columns="context",
                        values="mark")[primary_order]

display(primary_mat)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

rows = ["Early184","IFN6","Retinoid5","ECM13"]

# Panel A: discovery effects are directly comparable
A = traj[
    (traj.system=="Mouse discovery Müller") &
    (traj.signature.isin(rows))
].pivot(index="signature", columns="contrast", values="effect").loc[rows,["5D","15D"]]

# Panel B: independent validation contexts, direction only
val_order = primary_order[2:]
B = primary_mat.loc[rows,val_order]

fig, (ax1,ax2) = plt.subplots(
    1,2, figsize=(15,3.5),
    gridspec_kw={"width_ratios":[2,13]}
)

# A — quantitative discovery
im1 = ax1.imshow(A.values, cmap="RdBu_r", aspect="auto",
                 vmin=-A.abs().max().max(), vmax=A.abs().max().max())
for i in range(4):
    for j in range(2):
        ax1.text(j,i,f"{A.iloc[i,j]:.2f}",ha="center",va="center",fontsize=8)

ax1.set_xticks([0,1],["5D","15D"])
ax1.set_yticks(range(4),rows)
ax1.set_title("Discovery Müller")

# B — direction only
ax2.imshow(B.values, cmap=ListedColormap(["#3B6FB6","#F2F2F2","#C9473A"]),
           vmin=-1,vmax=1,aspect="auto")

ax2.set_xticks(range(len(val_order)))
ax2.set_xticklabels([x.replace(" | ","\n") for x in val_order],
                    rotation=55,ha="right",fontsize=8)
ax2.set_yticks(range(4),[""]*4)
ax2.set_title("Independent mouse and human systems — direction only")

plt.tight_layout()

fig.savefig(OUT/"Figure_integrated_trajectory_FINAL.svg", bbox_inches="tight")
fig.savefig(OUT/"Figure_integrated_trajectory_FINAL.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Independent validation contexts only
V = traj[
    ~traj["system"].eq("Mouse discovery Müller") &
    traj["signature"].isin(["Early184","IFN6","Retinoid5","ECM13"])
].copy()

V["positive"] = V["effect"] > 0
V["negative"] = V["effect"] < 0

summary = (
    V.groupby("signature")
     .agg(
         n_contexts=("effect","size"),
         n_positive=("positive","sum"),
         n_negative=("negative","sum"),
         median_effect=("effect","median")
     )
     .reset_index()
)

summary["fraction_positive"] = summary["n_positive"] / summary["n_contexts"]

summary.to_csv(
    OUT/"Locked_signature_cross_system_direction_summary.csv",
    index=False
)

display(summary.round(3))

In [ ]:
# Collapse correlated contrasts within each experimental arm
V["arm"] = V["system"].replace({
    "Mouse bulk":"Mouse bulk longitudinal",
    "Mouse Müller scRNA":"Mouse Müller 3mo",
    "Human Müller scRNA":"Human Müller atlas",
    "Human bulk retina":"Human bulk retina",
    "Human organoid":"Human organoid"
})

arm_summary = (
    V.groupby(["signature","arm"])
     .agg(
         n_contrasts=("effect","size"),
         median_effect=("effect","median"),
         positive_fraction=("positive","mean")
     )
     .reset_index()
)

# Directional label only; no cross-platform magnitude inference
arm_summary["direction"] = np.where(
    arm_summary["positive_fraction"] > 0.5, "mostly positive",
    np.where(arm_summary["positive_fraction"] < 0.5,
             "mostly negative", "mixed")
)

display(arm_summary[["signature","arm","n_contrasts",
                     "positive_fraction","direction"]])

In [ ]:
consensus = (
    arm_summary.groupby("signature")["direction"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

for c in ["mostly positive","mostly negative","mixed"]:
    if c not in consensus:
        consensus[c] = 0

consensus["n_arms"] = (
    consensus["mostly positive"] +
    consensus["mostly negative"] +
    consensus["mixed"]
)

consensus = consensus[
    ["signature","n_arms","mostly positive","mostly negative","mixed"]
]

consensus.to_csv(
    OUT/"Locked_signature_arm_level_consensus.csv",
    index=False
)

display(consensus)

In [ ]:
from scipy.stats import spearmanr

cols = ["early_effect","late_effect","bulk_1mo","bulk_3mo",
        "muller3mo","bulk_6mo","organoid"]

rho = pd.DataFrame(np.nan, index=cols, columns=cols)
nn  = pd.DataFrame(0, index=cols, columns=cols, dtype=int)

for i,a in enumerate(cols):
    for j,b in enumerate(cols):

        if a == b:
            n = M[a].notna().sum()
            rho.loc[a,b] = 1.0
            nn.loc[a,b] = n
        else:
            x = M[[a,b]].dropna()
            nn.loc[a,b] = len(x)
            rho.loc[a,b] = spearmanr(
                x[a].to_numpy(),
                x[b].to_numpy()
            ).statistic

rho.to_csv(OUT/"Early184_genelevel_Spearman_matrix.csv")
nn.to_csv(OUT/"Early184_genelevel_pairwise_N.csv")

display(rho.round(2))

In [ ]:
from itertools import combinations
from statsmodels.stats.multitest import multipletests

rows = []

for a,b in combinations(cols,2):
    x = M[[a,b]].dropna()
    r,p = spearmanr(x[a],x[b])
    rows.append([a,b,len(x),r,p])

corr_tests = pd.DataFrame(
    rows, columns=["context_A","context_B","n","rho","p"]
)

corr_tests["FDR_BH"] = multipletests(
    corr_tests["p"], method="fdr_bh"
)[1]

corr_tests.to_csv(
    OUT/"Early184_genelevel_Spearman_tests.csv",
    index=False
)

display(
    corr_tests.sort_values("p")
              .round(4)
)

In [ ]:
pairs = [
    ("bulk_1mo","bulk_3mo"),
    ("bulk_1mo","bulk_6mo"),
    ("bulk_3mo","bulk_6mo"),
    ("bulk_1mo","muller3mo"),
    ("bulk_3mo","muller3mo"),
    ("bulk_6mo","muller3mo"),
    ("early_effect","muller3mo"),
    ("early_effect","organoid")
]

rows = []
for a,b in pairs:
    x = M[[a,b]].dropna()
    same = np.sign(x[a]) == np.sign(x[b])
    rows.append([a,b,len(x),same.mean()])

sign_conc = pd.DataFrame(
    rows, columns=["context_A","context_B","n","fraction_same_direction"]
)

sign_conc.to_csv(
    OUT/"Early184_genelevel_sign_concordance.csv",
    index=False
)

display(sign_conc.round(3))

In [ ]:
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(42)
rows = []

for a,b in pairs:
    x = M[[a,b]].dropna()
    A = np.sign(x[a].to_numpy())
    B = np.sign(x[b].to_numpy())

    obs = np.mean(A == B)

    null = np.array([
        np.mean(A == rng.permutation(B))
        for _ in range(10000)
    ])

    center = null.mean()
    p = (1 + np.sum(np.abs(null-center) >= abs(obs-center))) / 10001

    rows.append([a,b,len(x),obs,center,p])

sign_test = pd.DataFrame(rows, columns=[
    "context_A","context_B","n",
    "observed_same","expected_same","empirical_p"
])

sign_test["FDR_BH"] = multipletests(
    sign_test["empirical_p"], method="fdr_bh"
)[1]

sign_test.to_csv(
    OUT/"Early184_sign_concordance_permutation_tests.csv",
    index=False
)

display(sign_test.round(4))

In [ ]:
rng = np.random.default_rng(42)
rows = []

# Mouse downstream contexts
for c in ["bulk_1mo","bulk_3mo","bulk_6mo","muller3mo"]:
    t = bg[bg.mouse_gene.isin(early_set) & bg[c].notna()]
    b = bg[~bg.mouse_gene.isin(early_set) & bg[c].notna()]

    be, bv = b.early_effect.to_numpy(), (b[c].to_numpy() > 0)

    cand = np.vstack([
        bv[np.argpartition(np.abs(be-e), 50)[:50]]
        for e in t.early_effect
    ])

    pick = rng.integers(0,50,size=(10000,len(t)))
    null = cand[np.arange(len(t))[None,:], pick].mean(axis=1)

    obs = (t[c] > 0).mean()
    center = null.mean()
    p = (1 + np.sum(np.abs(null-center) >= abs(obs-center))) / 10001
    rows.append([c,len(t),obs,center,p])

# Human organoid, using locked 150 target genes
t = locked_target
b = bg_org[~bg_org.mouse_gene.isin(early_set) & bg_org.organoid.notna()]

be, bv = b.early_effect.to_numpy(), (b.organoid.to_numpy() > 0)
cand = np.vstack([
    bv[np.argpartition(np.abs(be-e), 50)[:50]]
    for e in t.early_effect
])

pick = rng.integers(0,50,size=(10000,len(t)))
null = cand[np.arange(len(t))[None,:], pick].mean(axis=1)

obs = (t.organoid > 0).mean()
center = null.mean()
p = (1 + np.sum(np.abs(null-center) >= abs(obs-center))) / 10001
rows.append(["organoid",len(t),obs,center,p])

dir_null = pd.DataFrame(rows, columns=[
    "context","n","Early184_fraction_positive",
    "matched_expected_fraction","empirical_p"
])

dir_null["FDR_BH"] = multipletests(dir_null.empirical_p, method="fdr_bh")[1]

display(dir_null.round(4))

In [ ]:
pb = pd.read_csv(
    ROOT/"02_Human_scRNA_2026/Human_atlas_FROZEN_Muller_donor_pseudobulk_counts.csv.gz"
)
man = pd.read_csv(
    ROOT/"02_Human_scRNA_2026/Human_atlas_LOCKED_sample_donor_manifest.csv"
)

print("Pseudobulk:", pb.shape)
print(pb.columns.tolist()[:20])

print("\nManifest:", man.shape)
print(man.columns.tolist())
display(man.head())

In [ ]:
donor_meta = (
    man.groupby("donor")
       .agg(
           status=("status","first"),
           age=("age","first"),
           sex=("sex","first"),
           n_eyes=("sample","size")
       )
       .reset_index()
)

print(donor_meta.groupby("status").size())
display(donor_meta)

In [ ]:
gene_col = pb.columns[0]

print("Gene column:", gene_col)
display(pb.iloc[:5, :6])

X = pb.iloc[:,1:]

print("Genes:", len(pb))
print("Donors:", X.shape[1])
print("Negative values:", (X < 0).sum().sum())
print("Non-integer values:", ((X % 1) != 0).sum().sum())
print("Library sizes:")
print(X.sum().astype(int))

In [ ]:
# Median-ratio normalization across the 13 donor pseudobulks
C = pb.set_index(gene_col).astype(float)

gm = np.exp(np.log(C.replace(0,np.nan)).mean(axis=1))
ratios = C.div(gm, axis=0)
size_factors = ratios.replace([np.inf,-np.inf],np.nan).median(axis=0)

N = C.div(size_factors, axis=1)
L = np.log2(N + 1)

status = donor_meta.set_index("donor")["status"]

human_gene = pd.DataFrame({
    "human_gene": L.index,
    "DM_vs_NON": L[status[status=="DM"].index].mean(axis=1)
                 - L[status[status=="NON"].index].mean(axis=1),
    "DR_vs_NON": L[status[status=="DR"].index].mean(axis=1)
                 - L[status[status=="NON"].index].mean(axis=1),
    "DM_vs_DR":  L[status[status=="DM"].index].mean(axis=1)
                 - L[status[status=="DR"].index].mean(axis=1)
}).reset_index(drop=True)

human_gene.to_csv(
    OUT/"Human_Muller_donor_pseudobulk_gene_effects.csv",
    index=False
)

print(human_gene.shape)
display(human_gene.head())

In [ ]:
# Locked mouse→human mapping only
hm = (
    orth[["mouse_symbol","human_symbol"]]
    .merge(human_gene, left_on="human_symbol", right_on="human_gene")
    .groupby("mouse_symbol", as_index=False)
    [["DM_vs_NON","DR_vs_NON","DM_vs_DR"]].mean()
    .rename(columns={
        "mouse_symbol":"mouse_gene",
        "DM_vs_NON":"human_DM",
        "DR_vs_NON":"human_DR",
        "DM_vs_DR":"human_DM_vs_DR"
    })
)

M = M.merge(hm, on="mouse_gene", how="left")

print("Coverage:")
print(M[["human_DM","human_DR","human_DM_vs_DR"]].notna().sum())

# First comparison to discovery/chronic Müller
for c in ["human_DM","human_DR","human_DM_vs_DR"]:
    x = M[["early_effect","muller3mo",c]].dropna()
    r1,p1 = spearmanr(x["early_effect"],x[c])
    r2,p2 = spearmanr(x["muller3mo"],x[c])
    print(f"{c:15s} n={len(x):3d} | early rho={r1:+.3f} p={p1:.4g} | "
          f"mouse3mo rho={r2:+.3f} p={p2:.4g}")

In [ ]:
from statsmodels.stats.multitest import multipletests

rows = []

for h in ["human_DM","human_DR","human_DM_vs_DR"]:
    for m in ["early_effect","muller3mo"]:
        x = M[[m,h]].dropna()
        r,p = spearmanr(x[m],x[h])
        rows.append([m,h,len(x),r,p])

human_bridge = pd.DataFrame(
    rows, columns=["mouse_context","human_context","n","rho","p"]
)

human_bridge["FDR_BH"] = multipletests(
    human_bridge["p"], method="fdr_bh"
)[1]

human_bridge.to_csv(
    OUT/"Early184_mouse_human_Muller_bridge_correlations.csv",
    index=False
)

display(human_bridge.sort_values("p").round(4))

In [ ]:
import matplotlib.pyplot as plt

targets = ["late_effect","bulk_1mo","bulk_3mo","muller3mo",
           "bulk_6mo","organoid","human_DM","human_DR"]

labels = {
    "late_effect":"Mouse Müller 15D",
    "bulk_1mo":"Mouse bulk 1mo",
    "bulk_3mo":"Mouse bulk 3mo",
    "muller3mo":"Mouse Müller 3mo",
    "bulk_6mo":"Mouse bulk 6mo",
    "organoid":"Human organoid",
    "human_DM":"Human Müller DM",
    "human_DR":"Human Müller DR"
}

rows = []
for c in targets:
    x = M[["early_effect",c]].dropna()
    rows.append([labels[c], spearmanr(x.early_effect,x[c]).statistic])

fig, ax = plt.subplots(figsize=(7,4.5))
y = range(len(rows))

ax.barh(list(y), [r[1] for r in rows])
ax.set_yticks(list(y), [r[0] for r in rows])
ax.axvline(0, lw=1)
ax.set_xlim(-0.35,0.35)
ax.set_xlabel("Spearman ρ vs early 5D Müller effect")
ax.set_title("Early Müller gene ordering is not conserved downstream")
ax.invert_yaxis()

plt.tight_layout()
fig.savefig(OUT/"Figure_Early184_genelevel_concordance_FINAL.svg",
            bbox_inches="tight")
fig.savefig(OUT/"Figure_Early184_genelevel_concordance_FINAL.pdf",
            bbox_inches="tight")
plt.show()

In [ ]:
M.to_csv(OUT/"Early184_cross_system_gene_effects_MASTER.csv", index=False)
human_bridge.to_csv(OUT/"Human_Muller_bridge_FINAL.csv", index=False)
corr_tests.to_csv(OUT/"Early184_genelevel_correlations_FINAL.csv", index=False)
dir_null.to_csv(OUT/"Early184_matched_direction_tests_FINAL.csv", index=False)

print("CROSS-SYSTEM INTEGRATION CLOSED")
print("Early184 genes:", len(M))
print("Human Müller coverage:", M.human_DM.notna().sum())
print("Organoid coverage:", M.organoid.notna().sum())